# TS1 Substrate Frame Anchors in FRUST TS Guesses

Soft frame anchors are orientation handles. They are not optimizer constraints.

This notebook mirrors `d57_soft_frame_anchors.ipynb`, but focuses on TS1. TS1 has a single extra hydrogen fragment and no HBpin fragment, so the relevant soft frame is the substrate frame around `substrate_C`. A single hard role atom fixes where the reactive carbon should sit, but it does not fix how the aromatic substrate is rotated around that point. Nearby soft frame atoms give the rigid fragment placement enough local geometry to put the ring face in a chemically useful orientation.

The important distinction is:

| Anchor type | What it does during TS guess generation | What happens after placement |
| --- | --- | --- |
| Hard role atom | Defines the reactive TS core position, for example `cat_B`, `transfer_H`, or `substrate_C` | Snapped exactly onto the template coordinate |
| Soft frame atom | Helps the least-squares rigid fit orient the substrate fragment | Not snapped exactly; it can move after the hard anchors are restored |


## Build a small screen example

The example below uses the same public screen path users normally call: read components, expand them into substrate-catalyst systems, then create TS1 guesses. The notebook locally adds the TS1 substrate frame coordinates so the current TS1 guess can be compared against a hard-anchor-only TS1 guess.


In [13]:
import frust as ft
import numpy as np
import pandas as pd

import frust.tsguess.assembly as ts_assembly


In [14]:
TS1_SUBSTRATE_FRAME_COORDS = (
    (-0.310438, 1.801204, 0.223019),
    (-0.659429, 0.986440, 1.328243),
    (-1.274299, 1.891212, 2.230325),
)
ts_assembly.SUBSTRATE_ACTIVE_SITE_COORDS = {
    **ts_assembly.SUBSTRATE_ACTIVE_SITE_COORDS,
    "TS1": TS1_SUBSTRATE_FRAME_COORDS,
}

components = ft.screen.read(
    pd.DataFrame(
        {
            "role": ["substrate", "catalyst"],
            "smiles": ["C1=CC=CO1", "BC1=C(N(C)C)C=CC=C1"],
            "compound_name": ["furan", "dimethylaminophenyl_borane"],
            "rpos": [2, pd.NA],
        }
    )
)

systems = ft.screen.expand(components)
ts_guesses = ft.screen.create_ts_guesses(
    systems,
    ts_types=["TS1"],
    n_confs=3,
)

ts1 = ts_guesses["TS1"].reset_index(drop=True)
systems[["system_name", "substrate_name", "catalyst_name", "rpos"]]


,system_name,substrate_name,catalyst_name,rpos
0,furan__dimethylaminophenyl_borane,furan,dimethylaminophenyl_borane,2


## The TS1 guess with a substrate frame

This is the FRUST screen result after adding TS1 substrate frame coordinates in the notebook. The view below uses the same `ft.vis.ts_guess_scene(...)` viewer used for screen outputs.

The angle and distance guides are optimizer constraints from `constraint_spec`. Soft frame anchors are different: they are only used while orienting the disconnected fragments before the final TS guess dataframe is made.


In [15]:
ft.vis.show_scene(
    ft.vis.ts_guess_scene(
        ts1,
        row_indices=[0],
        show_roles=True,
        show_constraint_distances=True,
        show_constraint_angles=True,
        columns=1,
        cell_size=(760, 560),
    )
)


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

## Hard roles and soft frame atoms

The TS builder stores named reactive atoms in `constraint_roles`. Those role atoms are the hard anchors. The two substrate frame atoms are nearby atoms chosen from the assembled molecule. They are not named optimizer roles; they are temporary orientation handles used during rigid placement.

The helpers below inspect the generated row and build a visual overlay:

| Color | Meaning |
| --- | --- |
| Cyan | Hard TS role atom, snapped exactly after placement |
| Orange | Substrate soft frame atom, used to orient the substrate ring |


In [16]:
from collections.abc import Mapping
from contextlib import contextmanager

from tooltoad.scene3d import (
    AtomHighlight,
    AtomLabel,
    DistanceOverlay,
    GridScene,
    MoleculeModel,
    SceneCell,
)

HARD_ROLE_NAMES = ("cat_B", "cat_N", "transfer_H", "substrate_C")
VDW_RADII = {"H": 1.20, "B": 1.92, "C": 1.70, "N": 1.55, "O": 1.52}

def _row_bonds(row):
    bonds = row.get("connectivity_bonds", [])
    if bonds is None:
        return []
    return [(int(a), int(b)) for a, b in bonds]

def _bond_neighbors(row, atom_idx, *, symbols=None, exclude=()):
    exclude = {int(idx) for idx in exclude if idx is not None}
    neighbors = []
    for a, b in _row_bonds(row):
        if a == int(atom_idx) and b not in exclude:
            neighbors.append(b)
        elif b == int(atom_idx) and a not in exclude:
            neighbors.append(a)
    if symbols is not None:
        allowed = set(symbols)
        neighbors = [idx for idx in neighbors if str(row["atoms"][idx]) in allowed]
    return neighbors

def anchor_indices(row):
    roles = row.get("constraint_roles", {})
    if not isinstance(roles, Mapping):
        return {}, []

    hard = {role: int(roles[role]) for role in HARD_ROLE_NAMES if role in roles}

    substrate_soft = []
    if "substrate_C" in roles:
        substrate_soft = _bond_neighbors(row, roles["substrate_C"], exclude=[roles.get("cat_B")])
        substrate_soft = [idx for idx in substrate_soft if str(row["atoms"][idx]) != "H"][:2]

    return hard, substrate_soft

def anchor_cell(row, *, title, coords_col="coords_embedded"):
    hard, substrate_soft = anchor_indices(row)
    overlays = []

    for role, atom_idx in hard.items():
        overlays.extend(
            [
                AtomHighlight(atom=atom_idx, color="cyan", radius=0.54, alpha=0.35),
                AtomLabel(atom=atom_idx, text=f"hard: {role}"),
            ]
        )

    for number, atom_idx in enumerate(substrate_soft, start=1):
        overlays.extend(
            [
                AtomHighlight(atom=atom_idx, color="orange", radius=0.5, alpha=0.4),
                AtomLabel(atom=atom_idx, text=f"soft substrate {number}"),
            ]
        )

    return SceneCell(
        title=title,
        models=[
            MoleculeModel(
                atoms=row["atoms"],
                coords=row[coords_col],
                bonds=_row_bonds(row),
                style={"stick": {}, "sphere": {"radius": 0.28}},
            )
        ],
        overlays=overlays,
    )

def anchor_scene(df, *, row_index=0, title=None, coords_col="coords_embedded"):
    row = df.iloc[int(row_index)]
    if title is None:
        title = str(row.get("ts_type", "TS guess"))
    return GridScene(
        cells=[anchor_cell(row, title=title, coords_col=coords_col)],
        columns=1,
        cell_size=(760, 560),
        linked=False,
    )

def comparison_scene(current_df, no_soft_df, *, ts_type, row_index=0):
    return GridScene(
        cells=[
            anchor_cell(
                current_df.iloc[int(row_index)],
                title=f"{ts_type}: current hard + soft frame anchors",
            ),
            anchor_cell(
                no_soft_df.iloc[int(row_index)],
                title=f"{ts_type}: hard anchors only",
            ),
        ],
        columns=2,
        cell_size=(560, 520),
        linked=True,
    )

def _local_graph_pairs(row, *, max_depth=3):
    atoms = list(row["atoms"])
    adjacency = [set() for _ in atoms]
    for a, b in _row_bonds(row):
        adjacency[a].add(b)
        adjacency[b].add(a)

    excluded = set()
    for start in range(len(atoms)):
        seen = {start}
        frontier = {start}
        for _ in range(max_depth):
            next_frontier = set()
            for atom_idx in frontier:
                next_frontier.update(adjacency[atom_idx])
            next_frontier -= seen
            excluded.update(tuple(sorted((start, atom_idx))) for atom_idx in next_frontier)
            seen.update(next_frontier)
            frontier = next_frontier
    return excluded

def close_contact_records(row, *, cutoff=0.58, limit=None):
    atoms = list(row["atoms"])
    coords = np.asarray(row["coords_embedded"], dtype=float)
    excluded = _local_graph_pairs(row, max_depth=3)

    roles = row.get("constraint_roles", {}) or {}
    for constraint in row.get("constraint_spec", []) or []:
        if not isinstance(constraint, dict) or constraint.get("kind") != "distance":
            continue
        constraint_roles = constraint.get("roles") or []
        if len(constraint_roles) < 2:
            continue
        left, right = constraint_roles[:2]
        if left in roles and right in roles:
            excluded.add(tuple(sorted((int(roles[left]), int(roles[right])))))

    contacts = []
    for i, atom_i in enumerate(atoms):
        for j, atom_j in enumerate(atoms[i + 1 :], start=i + 1):
            if (i, j) in excluded:
                continue
            distance = float(np.linalg.norm(coords[i] - coords[j]))
            radius_sum = VDW_RADII.get(str(atom_i), 1.70) + VDW_RADII.get(str(atom_j), 1.70)
            normalized = distance / radius_sum
            if normalized < cutoff:
                contacts.append(
                    {
                        "atom1": i,
                        "atom2": j,
                        "symbols": f"{atom_i}-{atom_j}",
                        "distance_A": distance,
                        "normalized_vdw": normalized,
                    }
                )

    contacts.sort(key=lambda record: record["normalized_vdw"])
    if limit is not None:
        contacts = contacts[: int(limit)]
    return contacts

def close_contact_score(row, *, cutoff=0.58):
    contacts = close_contact_records(row, cutoff=cutoff)
    score = sum((cutoff - record["normalized_vdw"]) ** 2 for record in contacts)
    return score, len(contacts)

def clash_cell(row, *, title, coords_col="coords_embedded", cutoff=0.58, max_pairs=12):
    cell = anchor_cell(row, title=title, coords_col=coords_col)
    overlays = list(cell.overlays or [])
    for record in close_contact_records(row, cutoff=cutoff, limit=max_pairs):
        overlays.append(
            DistanceOverlay(
                atom1=int(record["atom1"]),
                atom2=int(record["atom2"]),
                label=f"{record['distance_A']:.2f} A",
                color="red",
                radius=0.045,
            )
        )
    return SceneCell(title=cell.title, models=cell.models, overlays=overlays)


In [17]:
ft.vis.show_scene(anchor_scene(ts1, row_index=0, title="TS1 hard roles and substrate frame atoms"))


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

The cyan atoms define the named TS core and are the atoms FRUST snaps exactly. The orange atoms are only there to give the rigid fragment fitting step enough geometry to know which face or local plane should point into the TS core.


## Mental model: one point, one axis, one plane

Rigid placement uses the coordinates in the placement map as matching points. More points remove more rotational freedom:

| Placement map | What it fixes | What remains ambiguous |
| --- | --- | --- |
| One anchor | Position | Rotation around that point |
| Two anchors | Position plus an axis | Rotation around that axis |
| Three non-collinear anchors | Position, axis, and local plane | The least-squares fit has a defined local orientation |

For TS1 substrate placement, `substrate_C` is the point and the two neighboring heavy atoms define the local ring frame.


In [18]:
def toy_anchor_cell(title, points, labels):
    atoms = ["C"] * len(points)
    overlays = []
    for idx, label in enumerate(labels):
        overlays.extend(
            [
                AtomHighlight(atom=idx, color="cyan", radius=0.5, alpha=0.35),
                AtomLabel(atom=idx, text=label),
            ]
        )
    return SceneCell(
        title=title,
        models=[
            MoleculeModel(
                atoms=atoms,
                coords=np.asarray(points, dtype=float),
                bonds=[(idx, idx + 1) for idx in range(len(points) - 1)],
                style={"stick": {"radius": 0.08}, "sphere": {"radius": 0.32}},
            )
        ],
        overlays=overlays,
    )

toy_scene = GridScene(
    cells=[
        toy_anchor_cell("1 anchor: position only", [[0, 0, 0]], ["point"]),
        toy_anchor_cell("2 anchors: local axis", [[-0.8, 0, 0], [0.8, 0, 0]], ["A", "B"]),
        toy_anchor_cell(
            "3 anchors: local plane",
            [[-0.8, 0, 0], [0.8, 0, 0], [0.0, 0.9, 0.25]],
            ["A", "B", "C"],
        ),
    ],
    columns=3,
    cell_size=(330, 300),
    linked=False,
)
ft.vis.show_scene(toy_scene)

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

## What happens if the substrate frame anchors are removed?

The code below is a local demonstration. It temporarily removes the TS1 substrate frame coordinate map, then regenerates the same TS guesses. This is not recommended production code; it is just a compact way to see what the soft frame atoms are doing.

With hard anchors only, the reactive atoms still land in roughly the right TS core positions. The problem is that the surrounding substrate orientation is underdetermined. Rotate the linked viewers and inspect the substrate ring face and close contacts near the reactive center.


In [19]:
@contextmanager
def without_soft_frame_anchors():
    original_substrate_frames = ts_assembly.SUBSTRATE_ACTIVE_SITE_COORDS
    try:
        ts_assembly.SUBSTRATE_ACTIVE_SITE_COORDS = {
            key: value
            for key, value in original_substrate_frames.items()
            if key != "TS1"
        }
        yield
    finally:
        ts_assembly.SUBSTRATE_ACTIVE_SITE_COORDS = original_substrate_frames

with without_soft_frame_anchors():
    no_soft_ts_guesses = ft.screen.create_ts_guesses(
        systems,
        ts_types=["TS1"],
        n_confs=3,
    )

no_soft_ts1 = no_soft_ts_guesses["TS1"].reset_index(drop=True)


In [20]:
ft.vis.show_scene(comparison_scene(ts1, no_soft_ts1, ts_type="TS1", row_index=0))


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

## A more catastrophic hard-only stress test

The furan example is intentionally small, so the hard-only result can look only mildly wrong. To make the failure mode obvious, use a bulkier substrate and catalyst, generate more hard-only conformers, and pick the hard-only TS1 row with the worst severe nonbonded contacts.

This is a diagnostic stress test, not a recommended workflow. The red guides below mark atom pairs that are much closer than their van der Waals radii would allow, after excluding ordinary bonded neighborhoods and the intended reactive distance constraints.


In [21]:
stress_components = ft.screen.read(
    pd.DataFrame(
        {
            "role": ["substrate", "catalyst"],
            "smiles": [
                "CC(C)(C)C1=CC=CC=C1",
                "BC1=C(N2CCCCC2)C=CC=C1",
            ],
            "compound_name": ["tertbutyl_benzene", "piperidine_cat"],
            "rpos": [5, pd.NA],
        }
    )
)
stress_systems = ft.screen.expand(stress_components)
stress_ts1 = ft.screen.create_ts_guesses(
    stress_systems,
    ts_types=["TS1"],
    n_confs=16,
)["TS1"].reset_index(drop=True)

with without_soft_frame_anchors():
    stress_no_soft_ts1 = ft.screen.create_ts_guesses(
        stress_systems,
        ts_types=["TS1"],
        n_confs=16,
    )["TS1"].reset_index(drop=True)

stress_systems[["system_name", "substrate_name", "catalyst_name", "rpos"]]


,system_name,substrate_name,catalyst_name,rpos
0,tertbutyl_benzene__piperidine_cat,tertbutyl_benzene,piperidine_cat,5


In [22]:
hard_only_scores = [
    (row_index, *close_contact_score(row))
    for row_index, row in stress_no_soft_ts1.iterrows()
]
stress_worst_no_soft_idx, stress_worst_score, stress_worst_n_contacts = max(
    hard_only_scores,
    key=lambda item: (item[1], item[2]),
)

current_contacts = close_contact_records(stress_ts1.iloc[0])
hard_only_contacts = close_contact_records(
    stress_no_soft_ts1.iloc[stress_worst_no_soft_idx]
)
stress_clash_summary = pd.DataFrame(
    [
        {
            "case": "current substrate-frame TS1 row 0",
            "row": 0,
            "severe_contacts": len(current_contacts),
            "worst_distance_A": min(
                (record["distance_A"] for record in current_contacts),
                default=pd.NA,
            ),
        },
        {
            "case": "hard-only TS1 worst sampled row",
            "row": stress_worst_no_soft_idx,
            "severe_contacts": len(hard_only_contacts),
            "worst_distance_A": min(
                (record["distance_A"] for record in hard_only_contacts),
                default=pd.NA,
            ),
        },
    ]
)
stress_clash_summary


,case,row,severe_contacts,worst_distance_A
0,current substrate-frame TS1 row 0,0,7,1.213612
1,hard-only TS1 worst sampled row,13,18,0.718232


In [23]:
pd.DataFrame(hard_only_contacts[:10])

,atom1,atom2,symbols,distance_A,normalized_vdw
0,15,48,H-H,0.718232,0.299263
1,35,52,C-H,1.133341,0.390807
2,14,47,H-H,0.962400,0.401000
3,4,48,C-H,1.181782,0.407511
4,14,32,H-C,1.182051,0.407604
5,28,41,H-H,0.987423,0.411426
6,2,52,C-H,1.353789,0.466824
7,14,30,H-C,1.384117,0.477282
8,0,47,B-H,1.497194,0.479870
9,48,52,H-H,1.156785,0.481994


In [24]:
stress_scene = GridScene(
    cells=[
        clash_cell(
            stress_ts1.iloc[0],
            title="Current substrate-frame TS1, row 0",
            max_pairs=12,
        ),
        clash_cell(
            stress_no_soft_ts1.iloc[stress_worst_no_soft_idx],
            title=f"Hard-only TS1, worst sampled row {stress_worst_no_soft_idx}",
            max_pairs=12,
        ),
    ],
    columns=2,
    cell_size=(600, 560),
    linked=True,
)
ft.vis.show_scene(stress_scene)


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

## How this connects to the implementation

The screen TS path separates identity from orientation:

1. `constraint_roles` gives named reactive atoms such as `cat_B`, `transfer_H`, and `substrate_C`.
2. The placement coordinate map adds those hard role atoms plus nearby soft frame atoms for substrate orientation.
3. Fragment placement uses a least-squares rigid transform from source anchors to template target coordinates.
4. After placement, FRUST snaps only the hard role atoms back to their exact role coordinates.
5. The optimizer constraints come from `constraint_spec`, not from the soft frame atoms.

Historically, `transformers.py` solved a related problem by aligning an incoming substrate onto a template substrate map with `rdMolAlign.AlignMol(...)`. That grafting-style path tied orientation to a particular template structure. The screen TS path keeps the same geometric idea but makes it more explicit: named TS roles define the reactive core, while soft frame atoms orient attached fragments during the TS guess assembly.


## Summary table

| Concept | Old transformer path | TS1 screen TS path shown here |
| --- | --- | --- |
| Reactive atom identity | Fixed or positional `keep_idxs` | Named `constraint_roles` |
| Substrate orientation | `rdMolAlign.AlignMol` to template substrate atoms | Substrate soft frame anchors |
| Optimizer constraints | Positional constraint atoms | `constraint_spec` role distances and angles |
| Exact snapping | Template core or coordMap behavior | Hard role atoms only |

The practical rule is: hard anchors say where the reactive atoms must be; soft frame anchors say which way the fragment should face while it is being placed.
